
# Workshop 1 — Source Profiling

**Team 8 — Tsunami Monitoring** · Data Analysis Programming 2026-II

This notebook profiles the four data sources of the project using the raw samples stored in `workshop_1/samples/`.

**Scope:** Pacific basin (HazEL regions 80–89 and the USGS rectangle lat −60..65, lon 100..300), 2010–2025, earthquakes M ≥ 6.

| Role | Source | Format | Samples used |
|---|---|---|---|
| Historical 1 | NOAA/NCEI HazEL tsunami events | JSON (REST) | `noaa_tsunamis_*.json`, `noaa_descriptor_*.json` |
| Historical 2 | USGS FDSN earthquake catalog | CSV | `usgs_earthquakes_*.csv` |
| Incremental 1 | USGS real-time feed (M4.5+, past month) | GeoJSON | `usgs_realtime_*.geojson` |
| Incremental 2 | tsunami.gov warning centers (NTWC, PTWC) | Atom + CAP | `ntwc_*.xml`, `ptwc_*.xml`, `*_cap*.xml` |

Sections: 1 Setup · 2 Load · 3 Profiling · 4 Coordinates, timestamps, units · 5 Data-quality problems · 6 Cross-source overlap · 7 Conclusions


## 1. Setup

In [ ]:

import json
import math
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 80)

SAMPLES = Path("../workshop_1/samples")
print("Samples folder:", SAMPLES.resolve())
for path in sorted(SAMPLES.iterdir()):
    if path.name != ".gitkeep":
        print(f"{path.stat().st_size:>10,} bytes  {path.name}")


## 2. Load the samples

### 2.1 NOAA HazEL — tsunami events (historical)

In [ ]:

noaa_files = sorted(f for f in SAMPLES.glob("noaa_tsunamis*.json") if "2011" not in f.name)
print("Files:", [f.name for f in noaa_files])

noaa = pd.DataFrame(
    [item for f in noaa_files for item in json.loads(f.read_text(encoding="utf-8"))["items"]]
)
print("Rows read:", len(noaa), "| duplicated ids:", noaa["id"].duplicated().sum())

def load_descriptor(name):
    items = json.loads((SAMPLES / f"noaa_descriptor_{name}.json").read_text(encoding="utf-8"))["items"]
    return {int(i["id"]): i["description"] for i in items}

causes = load_descriptor("causes")
validity = load_descriptor("validity")
warning_statuses = load_descriptor("warning_statuses")
regions = load_descriptor("regions")

noaa["cause"] = noaa["causeCode"].map(causes)
noaa["validity"] = noaa["eventValidity"].map(validity)
noaa["warning_status"] = noaa["warningStatusId"].map(warning_statuses)
noaa["region"] = noaa["regionCode"].map(regions)

# HazEL stores the date in six separate columns: build one UTC timestamp.
# Hour/minute/second can be missing; keep a flag instead of inventing values.
noaa["time_complete"] = noaa["hour"].notna() & noaa["minute"].notna()
noaa["time_utc"] = (
    pd.to_datetime(noaa[["year", "month", "day"]], errors="coerce", utc=True)
    + pd.to_timedelta(noaa["hour"].fillna(0), unit="h")
    + pd.to_timedelta(noaa["minute"].fillna(0), unit="m")
    + pd.to_timedelta(noaa["second"].fillna(0), unit="s")
)
print("Unparseable dates:", noaa["time_utc"].isna().sum())
noaa[["id", "time_utc", "time_complete", "latitude", "longitude", "eqMagnitude", "cause", "validity", "region"]].head()


### 2.2 USGS FDSN catalog — earthquakes (historical)

In [ ]:

usgs_file = sorted(SAMPLES.glob("usgs_earthquakes_*.csv"))[-1]
print("File:", usgs_file.name)

usgs_hist = pd.read_csv(usgs_file)
usgs_hist["time_utc"] = pd.to_datetime(usgs_hist["time"], utc=True)
usgs_hist["updated_utc"] = pd.to_datetime(usgs_hist["updated"], utc=True)
print("Rows:", len(usgs_hist), "| period:", usgs_hist["time_utc"].min(), "->", usgs_hist["time_utc"].max())
usgs_hist.head()


### 2.3 USGS real-time feed — earthquakes (incremental)

In [ ]:

rt_files = sorted(SAMPLES.glob("usgs_realtime_*.geojson"))
print("Snapshots available:", [f.name for f in rt_files])
rt_file = rt_files[-1]  # profile the latest snapshot
feed = json.loads(rt_file.read_text(encoding="utf-8"))
print("Feed title:", feed["metadata"]["title"], "| count:", feed["metadata"]["count"])
print("Generated:", pd.to_datetime(feed["metadata"]["generated"], unit="ms", utc=True))

usgs_rt = pd.json_normalize(feed["features"])
usgs_rt.columns = [c.replace("properties.", "") for c in usgs_rt.columns]
# GeoJSON coordinates are [longitude, latitude, depth] (longitude first)
usgs_rt[["lon", "lat", "depth"]] = pd.DataFrame(usgs_rt["geometry.coordinates"].tolist(), index=usgs_rt.index)
usgs_rt = usgs_rt.drop(columns=["geometry.coordinates", "geometry.type", "type"])
usgs_rt["time_utc"] = pd.to_datetime(usgs_rt["time"], unit="ms", utc=True)
usgs_rt["updated_utc"] = pd.to_datetime(usgs_rt["updated"], unit="ms", utc=True)
usgs_rt[["id", "time_utc", "updated_utc", "mag", "magType", "status", "tsunami", "lat", "lon", "depth", "place"]].head()


### 2.4 tsunami.gov — Atom feeds and CAP messages (incremental)

In [ ]:

NS = {
    "a": "http://www.w3.org/2005/Atom",
    "geo": "http://www.w3.org/2003/01/geo/wgs84_pos#",
    "cap": "urn:oasis:names:tc:emergency:cap:1.2",
}

def parse_cap(path):
    """Flatten a CAP message into one dictionary."""
    root = ET.parse(path).getroot()
    info = root.find("cap:info", NS)
    row = {
        "identifier": root.findtext("cap:identifier", namespaces=NS),
        "sent": root.findtext("cap:sent", namespaces=NS),
        "status": root.findtext("cap:status", namespaces=NS),
        "msgType": root.findtext("cap:msgType", namespaces=NS),
    }
    for tag in ["event", "urgency", "severity", "certainty", "headline"]:
        row[tag] = info.findtext(f"cap:{tag}", namespaces=NS)
    for param in info.findall("cap:parameter", NS):
        row[param.findtext("cap:valueName", namespaces=NS)] = param.findtext("cap:value", namespaces=NS)
    return row

def read_center(center):
    """Read the latest Atom snapshot of a warning center, plus the CAP of each entry."""
    atom_file = sorted(SAMPLES.glob(f"{center}_2*.xml"))[-1]  # excludes *_cap*.xml
    stamp = atom_file.stem.split(f"{center}_")[1]
    root = ET.parse(atom_file).getroot()
    rows = []
    for number, entry in enumerate(root.findall("a:entry", NS), start=1):
        row = {
            "center": center,
            "snapshot": stamp,
            "entry_id": entry.findtext("a:id", namespaces=NS),
            "title": entry.findtext("a:title", namespaces=NS),
            "updated": entry.findtext("a:updated", namespaces=NS),
            "lat": entry.findtext("geo:lat", namespaces=NS),
            "lon": entry.findtext("geo:long", namespaces=NS),
        }
        cap_file = SAMPLES / f"{center}_cap{number}_{stamp}.xml"
        if cap_file.exists():
            row.update({f"cap_{k}": v for k, v in parse_cap(cap_file).items()})
        rows.append(row)
    print(f"{center}: {atom_file.name} -> {len(rows)} entries")
    return pd.DataFrame(rows)

tsunami_gov = pd.concat([read_center("ntwc"), read_center("ptwc")], ignore_index=True)
tsunami_gov.T



## 3. Profiling

For every source: row count, duplicated identifiers, null percentage per field, data type and value ranges.
The workshop requires **at least 200 records per source, or the full source if it is smaller**.


In [ ]:

def profile(df, name, id_col):
    print(f"{name}: {len(df)} rows x {df.shape[1]} columns")
    print(f"  duplicated {id_col}: {df[id_col].duplicated().sum()}")
    print("  meets the 200-record requirement:", "yes" if len(df) >= 200 else "NO -> widen the sample unless this is the full source")
    table = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_count": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
    })
    numeric = df.select_dtypes("number")
    table["min"] = numeric.min()
    table["max"] = numeric.max()
    return table.sort_values("null_%", ascending=False)


### 3.1 NOAA HazEL events

In [ ]:
profile(noaa, 'NOAA HazEL tsunami events', 'id')

### 3.2 USGS FDSN catalog

In [ ]:
profile(usgs_hist, 'USGS FDSN catalog', 'id')

### 3.3 USGS real-time feed

In [ ]:
profile(usgs_rt, 'USGS real-time feed', 'id')

### 3.4 tsunami.gov

In [ ]:

print("Entries per center (each Atom feed only keeps the latest message):")
print(tsunami_gov.groupby("center").size())
profile(tsunami_gov, "tsunami.gov (NTWC + PTWC)", "entry_id")


## 4. Coordinates, timestamps and units

### 4.1 Coordinate validity (WGS84: latitude −90..90, longitude −180..180)

In [ ]:

def check_coords(df, lat, lon, name):
    lat_v = pd.to_numeric(df[lat], errors="coerce")
    lon_v = pd.to_numeric(df[lon], errors="coerce")
    return {
        "source": name,
        "rows": len(df),
        "missing": int((lat_v.isna() | lon_v.isna()).sum()),
        "lat_out_of_range": int(((lat_v < -90) | (lat_v > 90)).sum()),
        "lon_out_of_range": int(((lon_v < -180) | (lon_v > 180)).sum()),
        "at_0_0": int(((lat_v == 0) & (lon_v == 0)).sum()),
    }

pd.DataFrame([
    check_coords(noaa, "latitude", "longitude", "NOAA HazEL"),
    check_coords(usgs_hist, "latitude", "longitude", "USGS FDSN"),
    check_coords(usgs_rt, "lat", "lon", "USGS real-time"),
    check_coords(tsunami_gov, "lat", "lon", "tsunami.gov"),
])


### 4.2 Timestamp formats and time zones

In [ ]:

print("NOAA date columns (year, month, day, hour, minute, second), example:")
print(noaa.loc[noaa["time_complete"], ["year", "month", "day", "hour", "minute", "second"]].head(2).to_string(index=False))
print("USGS CSV time (ISO 8601, UTC):", usgs_hist["time"].iloc[0])
print("USGS real-time time (epoch milliseconds, UTC):", usgs_rt["time"].iloc[0], "->", usgs_rt["time_utc"].iloc[0])
print("tsunami.gov Atom updated (ISO 8601, UTC):", tsunami_gov["updated"].iloc[0])
if "cap_EventOriginTime" in tsunami_gov:
    print("tsunami.gov CAP EventOriginTime:", tsunami_gov["cap_EventOriginTime"].dropna().iloc[0])

ranges = pd.DataFrame({
    "first": [noaa["time_utc"].min(), usgs_hist["time_utc"].min(), usgs_rt["time_utc"].min()],
    "last": [noaa["time_utc"].max(), usgs_hist["time_utc"].max(), usgs_rt["time_utc"].max()],
}, index=["NOAA HazEL", "USGS FDSN", "USGS real-time"])
print("Time coverage of each sample:")
ranges


### 4.3 Units and severity scales

In [ ]:

print("USGS FDSN magnitude types:")
print(usgs_hist["magType"].value_counts().to_string())
print("USGS real-time magnitude types:")
print(usgs_rt["magType"].value_counts().to_string())

units = pd.DataFrame([
    ["NOAA HazEL", "eqMagnitude", "magnitude (type not given)", "eqDepth: km (verify in HazEL docs)"],
    ["USGS FDSN / real-time", "mag + magType", "mww, mb, mwc, ms, ml ...", "depth: km"],
    ["tsunami.gov Atom/CAP", "Preliminary Magnitude / EventPreliminaryMagnitude", "Ml, Mw ... (preliminary)", "CAP EventDepth: km; text bulletin: miles"],
], columns=["source", "severity field", "magnitude type", "depth unit"])
units



## 5. Data-quality problems

Each problem is computed from the samples, with one example record.


In [ ]:

problems = []

def add(problem, source, evidence, example):
    problems.append({"problem": problem, "source": source, "evidence": evidence, "example": example})

# 1. NOAA: official warning status missing for most events
missing = noaa["warningStatusId"].isna()
add("Official warning status missing", "NOAA HazEL",
    f"{missing.sum()} of {len(noaa)} events ({missing.mean():.0%}) have no warningStatusId",
    f"event id {noaa.loc[missing, 'id'].iloc[0]}")

# 2. NOAA: doubtful / erroneous events
bad = noaa[noaa["eventValidity"] <= 2]
add("Erroneous or doubtful tsunami events", "NOAA HazEL",
    f"{len(bad)} events with validity <= 2: {bad['validity'].value_counts().to_dict()}",
    f"event id {bad['id'].iloc[0]} -> {bad['validity'].iloc[0]}")

# 3. NOAA: incomplete time of day
no_time = ~noaa["time_complete"]
add("Time of day missing", "NOAA HazEL",
    f"{no_time.sum()} events have no hour/minute (only the date is known)",
    f"event id {noaa.loc[no_time, 'id'].iloc[0]} on {noaa.loc[no_time, 'time_utc'].iloc[0].date()}")

# 4. NOAA: no coordinates
no_coord = noaa["latitude"].isna() | noaa["longitude"].isna()
add("Events without coordinates", "NOAA HazEL",
    f"{no_coord.sum()} events cannot be placed on a map",
    f"event id {noaa.loc[no_coord, 'id'].iloc[0]} ({noaa.loc[no_coord, 'locationName'].iloc[0]})")

# 5. NOAA: tsunamis that were not caused by an earthquake
non_eq = ~noaa["causeCode"].isin([1, 3, 4, 5])
add("Tsunamis without an earthquake cause", "NOAA HazEL",
    f"{non_eq.sum()} events: {noaa.loc[non_eq, 'cause'].value_counts().to_dict()}",
    f"event id {noaa.loc[non_eq, 'id'].iloc[0]} -> {noaa.loc[non_eq, 'cause'].iloc[0]}")

# 6. USGS CSV: columns that are completely empty
empty_cols = [c for c in usgs_hist.columns if usgs_hist[c].isna().all()]
add("Columns that are always empty", "USGS FDSN (CSV)",
    f"{len(empty_cols)} columns are 100% null: {empty_cols}",
    "every row")

# 7. USGS: identifiers with different formats
odd = usgs_hist[usgs_hist["net"] != "us"]
if len(odd):
    add("Identifiers from different networks", "USGS FDSN (CSV)",
        f"net values: {usgs_hist['net'].value_counts().to_dict()}",
        f"id {odd['id'].iloc[0]} (net={odd['net'].iloc[0]}), {odd['place'].iloc[0]}")

# 8. USGS real-time: the tsunami flag does not mean a tsunami
flagged = usgs_rt[usgs_rt["tsunami"] == 1]
add("`tsunami` flag is only a bulletin indicator", "USGS real-time",
    f"{len(flagged)} of {len(usgs_rt)} events have tsunami=1; the flag does not mean a tsunami happened",
    f"id {flagged['id'].iloc[0]} ({flagged['place'].iloc[0]})" if len(flagged) else "none in this snapshot")

# 9. tsunami.gov: location as free text, depth in different units
atom_text = tsunami_gov[["center", "title"]].head(2).to_dict("records")
add("Location as free text; units mixed", "tsunami.gov",
    "region is text ('75 miles SW of ...'); the text bulletin gives depth in miles, the CAP in km",
    str(atom_text[0]))

pd.DataFrame(problems)



## 6. Cross-source overlap

Matching rule used for this exploration (to be justified in Workshop 3):
two records describe the same event when their origin times differ by at most **15 minutes** and their epicenters are at most **100 km** apart.
When the NOAA record has no time of day, only the **date** must match (flagged as `date_only`).


In [ ]:

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

def norm_lon(lon):
    """USGS can return longitudes above 180 in a Pacific query; bring them to -180..180."""
    return ((lon + 180) % 360) - 180

def find_pairs(noaa_df, usgs_df, max_minutes=15, max_km=100):
    rows = []
    for _, n in noaa_df.dropna(subset=["latitude", "longitude"]).iterrows():
        window = pd.Timedelta(minutes=max_minutes) if n["time_complete"] else pd.Timedelta(days=1)
        close = usgs_df[(usgs_df["time_utc"] - n["time_utc"]).abs() <= window]
        best = None
        for _, u in close.iterrows():
            dist = haversine_km(n["latitude"], n["longitude"], u["latitude"], norm_lon(u["longitude"]))
            if dist <= max_km and (best is None or dist < best["distance_km"]):
                best = {
                    "noaa_id": n["id"], "usgs_id": u["id"],
                    "noaa_time": n["time_utc"], "usgs_time": u["time_utc"],
                    "dt_minutes": round((u["time_utc"] - n["time_utc"]).total_seconds() / 60, 1),
                    "distance_km": round(dist, 1),
                    "noaa_mag": n["eqMagnitude"], "usgs_mag": u["mag"], "usgs_magType": u["magType"],
                    "noaa_depth": n["eqDepth"], "usgs_depth": u["depth"],
                    "date_only": not n["time_complete"],
                    "place": u["place"],
                }
        if best:
            rows.append(best)
    return pd.DataFrame(rows)

# Compare only NOAA events inside the period covered by the USGS sample
start = usgs_hist["time_utc"].min().normalize()
end = usgs_hist["time_utc"].max().normalize() + pd.Timedelta(days=1)
noaa_period = noaa[
    (noaa["time_utc"] >= start) & (noaa["time_utc"] <= end) & noaa["causeCode"].isin([1, 3, 4, 5])
]
print(f"Period compared: {start.date()} -> {end.date()}")
print(f"NOAA earthquake-caused tsunamis in that period: {len(noaa_period)}")

pairs = find_pairs(noaa_period, usgs_hist)
print(f"Pairs found (NOAA -> USGS): {len(pairs)}")
pairs


In [ ]:

if len(pairs):
    summary = pd.DataFrame({
        "pairs": [len(pairs)],
        "median |dt| (min)": [pairs["dt_minutes"].abs().median()],
        "median distance (km)": [pairs["distance_km"].median()],
        "median |mag diff|": [(pairs["noaa_mag"] - pairs["usgs_mag"]).abs().median()],
    })
    display(summary)
    print("Pairs with the largest differences:")
    display(
        pairs.assign(mag_diff=(pairs["noaa_mag"] - pairs["usgs_mag"]).abs())
        .sort_values(["mag_diff", "distance_km"], ascending=False)
        .head(5)[["noaa_id", "usgs_id", "noaa_mag", "usgs_mag", "usgs_magType",
                  "noaa_depth", "usgs_depth", "distance_km", "dt_minutes"]]
    )


### 6.1 tsunami.gov bulletin vs USGS real-time feed

In [ ]:

caps = tsunami_gov.dropna(subset=["cap_EventOriginTime"]) if "cap_EventOriginTime" in tsunami_gov else pd.DataFrame()
rows = []
for _, t in caps.iterrows():
    t_time = pd.to_datetime(t["cap_EventOriginTime"], utc=True)
    t_lat, t_lon = (float(x) for x in t["cap_EventLatLon"].split()[0].split(","))
    close = usgs_rt[(usgs_rt["time_utc"] - t_time).abs() <= pd.Timedelta(minutes=15)]
    for _, u in close.iterrows():
        dist = haversine_km(t_lat, t_lon, u["lat"], u["lon"])
        if dist <= 100:
            rows.append({
                "center": t["center"], "bulletin": t["cap_identifier"], "usgs_id": u["id"],
                "bulletin_time": t_time, "usgs_time": u["time_utc"],
                "dt_seconds": round((u["time_utc"] - t_time).total_seconds(), 1),
                "distance_km": round(dist, 1),
                "bulletin_mag": f"{t['cap_EventPreliminaryMagnitude']} {t['cap_EventPreliminaryMagnitudeType']}",
                "usgs_mag": f"{u['mag']} {u['magType']}",
                "bulletin_depth": t["cap_EventDepth"], "usgs_depth_km": round(u["depth"], 1),
                "usgs_tsunami_flag": u["tsunami"],
                "bulletin_says": t["cap_severity"] + " / " + t["cap_certainty"],
            })
bulletin_pairs = pd.DataFrame(rows)
print("Bulletin/USGS pairs found:", len(bulletin_pairs), "(a message can fall outside the 30-day window of the feed)")
bulletin_pairs.T



## 7. Conclusions for the canonical mapping

Fill in with the team after reviewing the outputs above.

| Topic | Finding | Decision for the data contract |
|---|---|---|
| Timestamps | NOAA splits the date in 6 columns; USGS uses ISO or epoch ms | Store `timestamp` in UTC; keep a `time_precision` flag |
| Coordinates | USGS GeoJSON is [lon, lat, depth]; Pacific queries return lon > 180 | Normalize to WGS84 −180..180 |
| Severity | Magnitude type differs across sources (Ml, mw, mww, mb) | Store the value and its type |
| Duplicates | Identifiers are source-specific | Event key = (source, id); cross-source links go in `event_group` |
| Missing values | `warningStatusId` and time of day are often missing | Keep NULL, never impute; document per field |
| Change detection | USGS revises `updated`/`mag`; tsunami.gov keeps only the latest message | Watermark on USGS `updated`; poll tsunami.gov often |
